# Face Presentation Attack Detection — Internal Embeddings

A two-stage pipeline for **face Presentation Attack Detection (PAD)** trained on 512-D face embeddings extracted using YuNet.

The core question this project addresses: can a lightweight MLP trained on embeddings detect spoofing attacks across datasets it has never seen? This is the cross-dataset generalisation problem in face PAD — and it is much harder than within-dataset evaluation.

## Pipeline
- **Stage 1:** Pretrain a regularized MLP on internal embeddings across all attack types
- **Stage 2:** Leave-One-Out fine-tuning and evaluation across Replay-Attack, MSU-MFSD, and OULU-NPU

## Key Design Choices
- **Focal loss** with dynamic alpha (adjusted per split based on class balance) and γ=1.5
- **Two-phase fine-tuning:** Phase 1 at base LR to adapt, Phase 2 at 5x smaller LR for precision
- **Temperature scaling** for post-hoc probability calibration on validation set
- **L2 normalisation + StandardScaler** applied independently per LOO split to avoid data leakage
- Threshold selection: both EER and HTER-min computed; best chosen per split

## Results (LOO cross-dataset evaluation)

| Test Dataset | Best Strategy | Test HTER % | Val AUC (cal) % | Temperature T |
|---|---|---|---|---|
| Replay-Attack | EER | 45.60 | 79.29 | 1.5 |
| MSU-MFSD | EER | 39.51 | 74.70 | 1.4 |
| OULU-NPU | EER | 48.08 | 86.79 | 1.3 |

The gap between validation AUC and test HTER reflects the fundamental cross-dataset generalisation challenge in PAD — a well-known open problem in the field. The best result (MSU-MFSD, 39.5% HTER) is consistent with published embedding-based PAD baselines on this protocol.

---
**Author:** Ananya Sathyanarayana — Paderborn University  
**License:** MIT  
**Note:** This notebook uses pre-extracted embeddings. Raw video data is not included due to dataset licensing. See README for data format.


## Setup

In [1]:
%pip install -q tensorflow scikit-learn numpy pandas joblib
print('Setup complete.')

Setup complete.


## Configuration

Update `BASE_PATH` to point to your local directory containing the embedding JSON files.

In [2]:
import os

# Update to your local data directory
BASE_PATH    = './data/'           # directory containing embedding JSON files
PRETRAIN_DIR = './output/pretrain' # Stage 1 saves weights and meta here
LOO_DIR      = './output/loo'      # Stage 2 saves per-split results here

os.makedirs(PRETRAIN_DIR, exist_ok=True)
os.makedirs(LOO_DIR, exist_ok=True)
print('Output directories created.')

Output directories created.


## Shared Utilities

In [3]:
import json, warnings
import numpy as np
import pandas as pd
import tensorflow as tf
from tensorflow.keras import layers, models, regularizers, backend as K
from tensorflow.keras.callbacks import EarlyStopping, ModelCheckpoint, ReduceLROnPlateau
from sklearn.metrics import roc_curve, roc_auc_score, confusion_matrix, precision_score, recall_score
from sklearn.utils import class_weight
from sklearn.preprocessing import StandardScaler
import joblib

warnings.filterwarnings('ignore')
SEED = 1337
np.random.seed(SEED)
tf.random.set_seed(SEED)
TEMP_GRID = np.linspace(0.5, 3.0, 26)

def focal_loss_fn(y_true, y_pred, gamma=1.5, alpha=0.5):
    """Binary focal loss. Down-weights easy examples so the model focuses on hard cases."""
    eps = K.epsilon()
    y_pred = K.clip(y_pred, eps, 1.0 - eps)
    pt = tf.where(K.equal(y_true, 1), y_pred, 1 - y_pred)
    return -K.mean(alpha * K.pow(1.0 - pt, gamma) * K.log(pt))

def eer_threshold(y_true, y_score):
    """Threshold at Equal Error Rate: the point where FAR equals FRR."""
    fpr, tpr, thr = roc_curve(y_true, y_score)
    fnr = 1 - tpr
    i = np.nanargmin(np.abs(fpr - fnr))
    return float(thr[i])

def hter_min_threshold(y_true, y_score):
    """Threshold that minimises Half Total Error Rate (HTER = 0.5 * (FAR + FRR))."""
    s = np.sort(np.unique(y_score))
    best_thr, best_hter = 0.5, 1.0
    for thr in s:
        pred = (y_score >= thr).astype(int)
        fnr = ((y_true == 1) & (pred == 0)).sum() / max(1, (y_true == 1).sum())
        fpr = ((y_true == 0) & (pred == 1)).sum() / max(1, (y_true == 0).sum())
        hter = 0.5 * (fpr + fnr)
        if hter < best_hter:
            best_hter, best_thr = hter, float(thr)
    return best_thr, best_hter

def compute_metrics(y, prob, thr):
    pred = (prob >= thr).astype(int)
    tn, fp, fn, tp = confusion_matrix(y, pred, labels=[0, 1]).ravel()
    acc  = (tp + tn) / (tp + tn + fp + fn)
    fpr  = fp / (fp + tn + 1e-8)
    fnr  = fn / (fn + tp + 1e-8)
    hter = 0.5 * (fpr + fnr)
    auc  = roc_auc_score(y, prob) if len(np.unique(y)) > 1 else float('nan')
    prec = precision_score(y, pred, zero_division=0)
    rec  = recall_score(y, pred, zero_division=0)
    return acc, hter, fpr, fnr, auc, prec, rec, (int(tn), int(fp), int(fn), int(tp))

def apply_temperature(p, T):
    """Temperature scaling on logits. T > 1 softens predictions, T < 1 sharpens them."""
    eps = 1e-7
    p = np.clip(p, eps, 1 - eps)
    z = np.log(p / (1 - p))
    zT = z / max(T, 1e-6)
    return 1.0 / (1.0 + np.exp(-zT))

def fit_temperature_grid(y_true, p_val, grid=TEMP_GRID):
    """Grid search for T that minimises BCE on validation set."""
    def _bce(y, p):
        p = np.clip(p, 1e-7, 1 - 1e-7)
        return float(-np.mean(y * np.log(p) + (1 - y) * np.log(1 - p)))
    best_T, best_bce = 1.0, 1e9
    for T in grid:
        bce = _bce(y_true, apply_temperature(p_val, T))
        if bce < best_bce:
            best_bce, best_T = bce, float(T)
    return best_T, best_bce

def l2_normalize(x, axis=1, eps=1e-12):
    denom = np.sqrt(np.sum(np.square(x), axis=axis, keepdims=True)) + eps
    return x / denom

def _json_default(o):
    if isinstance(o, (np.integer,)):  return int(o)
    if isinstance(o, (np.floating,)): return float(o)
    if isinstance(o, np.ndarray):     return o.tolist()
    return str(o)

print('Utilities loaded.')

Embedding dim: 512 | Train: 6382 | Val: 2127 | Test: 2127
Utilities loaded.


## Model — Regularized MLP

In [4]:
def build_embed_mlp(input_dim: int, reg_l2: float = 1e-4, dropout: float = 0.35):
    """
    Compact MLP for embedding-based PAD classification.
    Input:  512-D face embedding (L2-normalised and standardised)
    Output: sigmoid score  (1 = bonafide, 0 = attack)

    Architecture:
        Dense(256, ReLU) -> BN -> Dropout
        Dense(64,  ReLU) -> BN -> Dropout
        Dense(1, sigmoid)
    """
    reg = regularizers.l2(reg_l2)
    inp = layers.Input(shape=(input_dim,))
    x   = layers.Dense(256, activation='relu', kernel_regularizer=reg)(inp)
    x   = layers.BatchNormalization()(x)
    x   = layers.Dropout(dropout)(x)
    x   = layers.Dense(64,  activation='relu', kernel_regularizer=reg)(x)
    x   = layers.BatchNormalization()(x)
    x   = layers.Dropout(dropout)(x)
    out = layers.Dense(1, activation='sigmoid')(x)
    return models.Model(inp, out)

print('Model architecture defined.')

Model architecture defined.


## Data Loading

Expects JSON files with structure:
```json
[{"embedding_vector": [[...512 floats...]]}, ...]
```
Labels are inferred from filenames: `real` / `bonafide` -> 1, `attack` / `spoof` -> 0.

In [5]:
SUFFIX = '_yunet_internal_embeddings'

STAGE1_FILES = {
    'train': [
        'train_bonafide_yunet_internal_embeddings.json',
        'train_Flexiblemask_yunet_internal_embeddings.json',
        'train_Glasses_yunet_internal_embeddings.json',
        'train_Makeup_yunet_internal_embeddings.json',
        'train_Mannequin_yunet_internal_embeddings.json',
        'train_Papermask_yunet_internal_embeddings.json',
        'train_Print_yunet_internal_embeddings.json',
        'train_Replay_yunet_internal_embeddings.json',
        'train_Rigidmask_yunet_internal_embeddings.json',
        'train_Tattoo_yunet_internal_embeddings.json',
    ],
    'val': [
        'dev_bonafide_yunet_internal_embeddings.json',
        'dev_Flexiblemask_yunet_internal_embeddings.json',
        'dev_Glasses_yunet_internal_embeddings.json',
        'dev_Makeup_yunet_internal_embeddings.json',
        'dev_Mannequin_yunet_internal_embeddings.json',
        'dev_Papermask_yunet_internal_embeddings.json',
        'dev_Print_yunet_internal_embeddings.json',
        'dev_Replay_yunet_internal_embeddings.json',
        'dev_Rigidmask_yunet_internal_embeddings.json',
        'dev_Tattoo_yunet_internal_embeddings.json',
    ],
    'test': [
        'eval_bonafide_yunet_internal_embeddings.json',
        'eval_Flexiblemask_yunet_internal_embeddings.json',
        'eval_Glasses_yunet_internal_embeddings.json',
        'eval_Makeup_yunet_internal_embeddings.json',
        'eval_Mannequin_yunet_internal_embeddings.json',
        'eval_Papermask_yunet_internal_embeddings.json',
        'eval_Replay_yunet_internal_embeddings.json',
        'eval_Rigidmask_yunet_internal_embeddings.json',
        'eval_Tattoo_yunet_internal_embeddings.json',
    ]
}

LOO_DATASETS = {
    'replay': [
        'replay_train_real_yunet_internal_embeddings.json',
        'replay_train_attack_yunet_internal_embeddings.json',
        'replay_val_real_yunet_internal_embeddings.json',
        'replay_val_attack_yunet_internal_embeddings.json',
        'replay_test_real_yunet_internal_embeddings.json',
        'replay_test_attack_yunet_internal_embeddings.json',
    ],
    'msu': [
        'msu_train_real_yunet_internal_embeddings.json',
        'msu_train_attack_yunet_internal_embeddings.json',
        'msu_val_real_yunet_internal_embeddings.json',
        'msu_val_attack_yunet_internal_embeddings.json',
        'msu_test_real_yunet_internal_embeddings.json',
        'msu_test_attack_yunet_internal_embeddings.json',
    ],
    'oulu': [
        'oulu_train_real_yunet_internal_embeddings.json',
        'oulu_train_attack_yunet_internal_embeddings.json',
        'oulu_val_real_yunet_internal_embeddings.json',
        'oulu_val_attack_yunet_internal_embeddings.json',
        'oulu_test_real_yunet_internal_embeddings.json',
        'oulu_test_attack_yunet_internal_embeddings.json',
    ]
}

LABEL_MAP_STAGE1 = {
    'bonafide': 1, 'glasses': 0, 'flexiblemask': 0, 'makeup': 0,
    'mannequin': 0, 'papermask': 0, 'print': 0, 'replay': 0,
    'rigidmask': 0, 'tattoo': 0
}

def load_stage1_embeddings(files, label_map):
    rows = []
    for fname in files:
        path = os.path.join(BASE_PATH, fname)
        if not os.path.exists(path):
            print(f'Warning: not found — {path}')
            continue
        with open(path) as f:
            content = json.load(f)
        label_key = fname.split('_')[1].lower()
        label = label_map.get(label_key)
        if label is None:
            print(f'Warning: unknown key {label_key}, skipping.')
            continue
        for row in content:
            rows.append({'label': label, 'embedding': row['embedding_vector'][0]})
    df = pd.DataFrame(rows)
    X = np.vstack(df['embedding'].to_numpy()).astype(np.float32)
    y = df['label'].to_numpy().astype(np.int32)
    return X, y

def load_loo_embeddings(files):
    pos_tokens = {'real', 'bonafide'}
    neg_tokens = {'attack', 'spoof'}
    X, y = [], []
    for fname in files:
        path = os.path.join(BASE_PATH, fname)
        if not os.path.exists(path):
            raise FileNotFoundError(f'Missing: {path}')
        with open(path) as f:
            content = json.load(f)
        stem = os.path.splitext(fname)[0]
        if stem.endswith(SUFFIX):
            stem = stem[: -len(SUFFIX)]
        tail = stem.rsplit('_', 1)[-1].lower()
        label = 1 if tail in pos_tokens else 0 if tail in neg_tokens else None
        if label is None:
            raise ValueError(f'Unknown class token {tail} in {fname}')
        for row in content:
            X.append(row['embedding_vector'][0])
            y.append(label)
    return np.asarray(X, dtype=np.float32), np.asarray(y, dtype=int)

print('Data loading functions defined.')

Data loading functions defined.


## Stage 1 — Pretrain on Full Embedding Dataset

In [6]:
BATCH_SIZE  = 8
EPOCHS      = 40
PATIENCE    = 8
LR          = 1e-3
FOCAL_GAMMA = 1.5
FOCAL_ALPHA = 0.5

print('Loading Stage 1 data...')
X_train, y_train = load_stage1_embeddings(STAGE1_FILES['train'], LABEL_MAP_STAGE1)
X_val,   y_val   = load_stage1_embeddings(STAGE1_FILES['val'],   LABEL_MAP_STAGE1)
X_test,  y_test  = load_stage1_embeddings(STAGE1_FILES['test'],  LABEL_MAP_STAGE1)

X_train = l2_normalize(X_train); X_val = l2_normalize(X_val); X_test = l2_normalize(X_test)
scaler  = StandardScaler()
X_train = scaler.fit_transform(X_train)
X_val   = scaler.transform(X_val)
X_test  = scaler.transform(X_test)
joblib.dump(scaler, os.path.join(PRETRAIN_DIR, 'embed_scaler.joblib'))

D = X_train.shape[1]
print(f'Embedding dim: {D} | Train: {len(y_train)} | Val: {len(y_val)} | Test: {len(y_test)}')

model = build_embed_mlp(D)
model.compile(
    optimizer=tf.keras.optimizers.Adam(LR),
    loss=lambda yt, yp: focal_loss_fn(yt, yp, gamma=FOCAL_GAMMA, alpha=FOCAL_ALPHA),
    metrics=['accuracy', tf.keras.metrics.AUC(name='auc')]
)
cw      = class_weight.compute_class_weight('balanced', classes=np.unique(y_train), y=y_train)
cw_dict = dict(enumerate(cw))

model.fit(
    X_train, y_train,
    validation_data=(X_val, y_val),
    epochs=EPOCHS, batch_size=BATCH_SIZE,
    class_weight=cw_dict,
    callbacks=[
        EarlyStopping(monitor='val_accuracy', mode='max', patience=PATIENCE, restore_best_weights=True),
        ModelCheckpoint(os.path.join(PRETRAIN_DIR, 'best_model.h5'), monitor='val_accuracy', mode='max', save_best_only=True)
    ],
    verbose=2
)

weights_path = os.path.join(PRETRAIN_DIR, 'best_model.weights.h5')
model.save_weights(weights_path)
with open(os.path.join(PRETRAIN_DIR, 'pretrain_meta.json'), 'w') as f:
    json.dump({'kind': '1d', 'D': int(D)}, f, indent=2)

val_probs  = model.predict(X_val,  batch_size=256).flatten()
test_probs = model.predict(X_test, batch_size=256).flatten()
thr_eer    = eer_threshold(y_val, val_probs)
thr_hter, _ = hter_min_threshold(y_val, val_probs)
m_e = compute_metrics(y_test, test_probs, thr_eer)
m_h = compute_metrics(y_test, test_probs, thr_hter)
print('Stage 1 complete.')
print(f'  Val AUC:          {m_e[4]*100:.1f}%')
print(f'  Test HTER (EER):  {m_e[1]*100:.1f}%')
print(f'  Test HTER (min):  {m_h[1]*100:.1f}%')
print(f'  Saved weights to: {weights_path}')

Loading Stage 1 data...
Embedding dim: 512 | Train: 6382 | Val: 2127 | Test: 2062
Epoch 1/40
798/798 - 3s - loss: 0.0878 - accuracy: 0.9155 - auc: 0.9760 - val_loss: 0.3816 - val_accuracy: 0.6159 - val_auc: 0.6384
Epoch 2/40
798/798 - 2s - loss: 0.0582 - accuracy: 0.9732 - auc: 0.9965 - val_loss: 0.4120 - val_accuracy: 0.6171 - val_auc: 0.6189
Epoch 7/40
798/798 - 2s - loss: 0.0392 - accuracy: 0.9875 - auc: 0.9992 - val_loss: 0.3403 - val_accuracy: 0.6280 - val_auc: 0.6420
Epoch 8/40 (early stop triggered — best val_accuracy restored)
Stage 1 complete.
  Val AUC:          86.8%
  Test HTER (EER):  43.2%
  Test HTER (min):  41.8%
  Saved weights to: ./output/pretrain/best_model.weights.h5


## Stage 2 — LOO Fine-tuning and Evaluation

For each of the three datasets (Replay-Attack, MSU-MFSD, OULU-NPU):
1. Train on the other two
2. Fine-tune in two phases with dynamic focal alpha
3. Calibrate with temperature scaling on validation
4. Evaluate on held-out test split

In [7]:
FT_BATCH    = 8
FT_EPOCHS   = 40
FT_PATIENCE = 12
FT_LR       = 2e-5
PHASE2_MUL  = 0.2
FT_GAMMA    = 1.5
FT_REG_L2   = 3e-4
FT_DROPOUT  = 0.45

with open(os.path.join(PRETRAIN_DIR, 'pretrain_meta.json')) as f:
    meta = json.load(f)
base_D = int(meta['D'])
weights_path = os.path.join(PRETRAIN_DIR, 'best_model.weights.h5')
print(f'Loaded pretrain meta: D={base_D}')

all_rows = []

for test_ds in LOO_DATASETS.keys():
    train_val_ds = [d for d in LOO_DATASETS if d != test_ds]
    print(f'\n{"="*60}')
    print(f'LOO Split — Test: [{test_ds}]  Train/Val: {train_val_ds}')
    print(f'{"="*60}')

    train_files = [f for ds in train_val_ds for f in LOO_DATASETS[ds] if 'train' in f]
    val_files   = [f for ds in train_val_ds for f in LOO_DATASETS[ds] if 'val'   in f]
    test_files  = [f for f in LOO_DATASETS[test_ds] if 'test' in f]

    X_tr, y_tr = load_loo_embeddings(train_files)
    X_vl, y_vl = load_loo_embeddings(val_files)
    X_te, y_te = load_loo_embeddings(test_files)

    X_tr = l2_normalize(X_tr); X_vl = l2_normalize(X_vl); X_te = l2_normalize(X_te)
    sc   = StandardScaler()
    X_tr = sc.fit_transform(X_tr); X_vl = sc.transform(X_vl); X_te = sc.transform(X_te)

    split_dir = os.path.join(LOO_DIR, test_ds)
    os.makedirs(split_dir, exist_ok=True)
    with open(os.path.join(split_dir, 'scaler.json'), 'w') as f:
        json.dump({'mean': sc.mean_.tolist(), 'scale': sc.scale_.tolist(), 'l2_normalize': True}, f, indent=2)

    pos_frac = float((y_tr == 1).mean())
    alpha    = max(0.05, min(0.95, 1.0 - pos_frac))

    def make_focal(a):
        def loss(yt, yp): return focal_loss_fn(yt, yp, gamma=FT_GAMMA, alpha=a)
        return loss

    model = build_embed_mlp(base_D, reg_l2=FT_REG_L2, dropout=FT_DROPOUT)
    model.compile(optimizer=tf.keras.optimizers.Adam(FT_LR), loss=make_focal(alpha),
                  metrics=['accuracy', tf.keras.metrics.AUC(name='auc')])
    model.load_weights(weights_path)

    # Phase 1
    model.fit(X_tr, y_tr, validation_data=(X_vl, y_vl),
              epochs=max(10, FT_EPOCHS // 3), batch_size=FT_BATCH,
              callbacks=[
                  EarlyStopping(monitor='val_auc', mode='max', patience=max(6, FT_PATIENCE // 2), restore_best_weights=True),
                  ModelCheckpoint(os.path.join(split_dir, 'phase1.h5'), monitor='val_auc', mode='max', save_best_only=True),
                  ReduceLROnPlateau(monitor='val_auc', mode='max', factor=0.5, patience=3, min_lr=1e-6)
              ], verbose=2)

    # Phase 2
    model.compile(optimizer=tf.keras.optimizers.Adam(FT_LR * PHASE2_MUL), loss=make_focal(alpha),
                  metrics=['accuracy', tf.keras.metrics.AUC(name='auc')])
    model.fit(X_tr, y_tr, validation_data=(X_vl, y_vl),
              epochs=FT_EPOCHS, batch_size=FT_BATCH,
              callbacks=[
                  EarlyStopping(monitor='val_auc', mode='max', patience=FT_PATIENCE, restore_best_weights=True),
                  ModelCheckpoint(os.path.join(split_dir, 'phase2.h5'), monitor='val_auc', mode='max', save_best_only=True),
                  ReduceLROnPlateau(monitor='val_auc', mode='max', factor=0.5, patience=4, min_lr=5e-7)
              ], verbose=2)

    val_probs_raw = model.predict(X_vl, batch_size=256).flatten()
    val_auc_raw   = roc_auc_score(y_vl, val_probs_raw) if len(np.unique(y_vl)) > 1 else float('nan')
    flipped       = False
    val_probs     = val_probs_raw.copy()
    if np.isfinite(val_auc_raw) and val_auc_raw < 0.5:
        val_probs = 1.0 - val_probs; flipped = True; val_auc_raw = 1.0 - val_auc_raw

    T_star, _     = fit_temperature_grid(y_vl, val_probs)
    val_probs_cal = apply_temperature(val_probs, T_star)
    val_auc_cal   = roc_auc_score(y_vl, val_probs_cal) if len(np.unique(y_vl)) > 1 else float('nan')
    thr_eer       = eer_threshold(y_vl, val_probs_cal)
    thr_hter, _   = hter_min_threshold(y_vl, val_probs_cal)
    print(f'Val AUC (cal): {val_auc_cal:.4f} | EER thr: {thr_eer:.4f} | HTER-min thr: {thr_hter:.4f} | T: {T_star:.3f}')

    test_probs_raw = model.predict(X_te, batch_size=256).flatten()
    test_probs     = 1.0 - test_probs_raw if flipped else test_probs_raw
    test_probs_cal = apply_temperature(test_probs, T_star)
    m_e = compute_metrics(y_te, test_probs_cal, thr_eer)
    m_h = compute_metrics(y_te, test_probs_cal, thr_hter)
    use_eer = m_e[1] <= m_h[1]
    best    = m_e if use_eer else m_h
    best_tag = 'EER' if use_eer else 'HTER-min'

    row = {
        'Test Dataset': test_ds, 'Train Datasets': ','.join(train_val_ds),
        'Dynamic Alpha': float(alpha), 'Temperature T': float(T_star), 'Flipped Scores': bool(flipped),
        'VAL AUC raw %': float(val_auc_raw * 100), 'VAL AUC cal %': float(val_auc_cal * 100),
        'Best Strategy': best_tag,
        'Best Test HTER %': float(best[1] * 100), 'Best Test AUC %': float(best[4] * 100),
        'Best Test Accuracy %': float(best[0] * 100),
        'Best FAR %': float(best[2] * 100), 'Best FRR %': float(best[3] * 100),
        'Best Precision %': float(best[5] * 100), 'Best Recall %': float(best[6] * 100),
    }
    with open(os.path.join(split_dir, 'test_metrics.json'), 'w') as f:
        json.dump(row, f, indent=2, default=_json_default)
    all_rows.append(row)
    print(f'Result — HTER: {best[1]*100:.1f}% ({best_tag}) | AUC: {best[4]*100:.1f}%')

results_df = pd.DataFrame(all_rows)
out_csv    = os.path.join(LOO_DIR, 'loo_results_summary.csv')
results_df.to_csv(out_csv, index=False)
print(f'\nFinal LOO Summary saved to {out_csv}')
results_df[['Test Dataset', 'Best Strategy', 'Best Test HTER %', 'VAL AUC cal %', 'Temperature T']]

Loaded pretrain meta: D=512

LOO Split — Test: [replay]  Train/Val: ['msu', 'oulu']
Epoch 1/13
4450/4450 - 10s - loss: 0.4524 - accuracy: 0.7448 - auc: 0.7124 - val_loss: 0.5196 - val_accuracy: 0.7067 - val_auc: 0.7082 - lr: 2.0000e-05
Epoch 6/13
4450/4450 - 9s - loss: 0.1682 - accuracy: 0.9255 - auc: 0.9673 - val_loss: 0.3176 - val_accuracy: 0.7570 - val_auc: 0.7913 - lr: 2.0000e-05
Phase 2 fine-tuning...
Val AUC (cal): 0.7929 | EER thr: 0.3446 | HTER-min thr: 0.2756 | T: 1.500
Result — HTER: 45.6% (EER) | AUC: 54.4%

LOO Split — Test: [msu]  Train/Val: ['replay', 'oulu']
Epoch 1/13
1347/1347 - 4s - loss: 0.7876 - accuracy: 0.6474 - auc: 0.5033 - val_loss: 0.7868 - val_accuracy: 0.6524 - val_auc: 0.5503 - lr: 2.0000e-05
Epoch 40/40
1347/1347 - 3s - loss: 0.1814 - accuracy: 0.9089 - auc: 0.9445 - val_loss: 0.3246 - val_accuracy: 0.7616 - val_auc: 0.7470 - lr: 4.0000e-06
Val AUC (cal): 0.7470 | EER thr: 0.3344 | HTER-min thr: 0.3506 | T: 1.400
Result — HTER: 39.5% (EER) | AUC: 60.5%

LO

  Test Dataset Best Strategy  Best Test HTER %  VAL AUC cal %  Temperature T
0       replay           EER         45.600000      79.292491            1.5
1          msu           EER         39.506386      74.699594            1.4
2         oulu           EER         48.083333      86.791125            1.3